In [43]:
#! pip install kafka-python

In [44]:
from kafka import KafkaAdminClient

In [45]:
# create connection to kafka broker 

admin= KafkaAdminClient(
    bootstrap_servers="localhost:9092"
)
print("connected successfully")

connected successfully


In [46]:
#list all topics 
topics = admin.list_topics()
print(topics)

['sensor-telemetry']


In [47]:
admin.describe_cluster()


{'throttle_time_ms': 0,
 'brokers': [{'node_id': 1, 'host': 'localhost', 'port': 9092, 'rack': None}],
 'cluster_id': '5L6g3nShT-eMCtK--X86sw',
 'controller_id': 1,
 'authorized_operations': ['CREATE',
  'ALTER',
  'DESCRIBE',
  'CLUSTER_ACTION',
  'DESCRIBE_CONFIGS',
  'ALTER_CONFIGS',
  'IDEMPOTENT_WRITE']}

In [48]:
from kafka.admin import NewTopic

In [49]:
# define topic configuration 
sensor_topic = NewTopic(
    name= "sensor-telemetry",
    num_partitions=3,
    replication_factor=1
)

In [50]:
# create  topic in kafka 
admin.create_topics(
    new_topics=[sensor_topic],
    validate_only=False
)

print("topics crreated sucessfully")

TopicAlreadyExistsError: [Error 36] TopicAlreadyExistsError: Request 'CreateTopicsRequest_v3(create_topic_requests=[(topic='sensor-telemetry', num_partitions=3, replication_factor=1, replica_assignment=[], configs=[])], timeout=30000, validate_only=False)' failed with response 'CreateTopicsResponse_v3(throttle_time_ms=0, topic_errors=[(topic='sensor-telemetry', error_code=36, error_message="Topic 'sensor-telemetry' already exists.")])'.

In [ ]:
# verify topic exists  or not
topics =admin.list_topics()
print(topics)


['sensor-telemetry']


In [ ]:
#inspect topic metadata
from kafka import KafkaConsumer
consumer = KafkaConsumer(
    bootstrap_servers= "localhost:9092"
)

partitions= consumer.partitions_for_topic(
    "sensor-telemetry"
)
print(partitions)

{0, 1, 2}


In [ ]:
# import kafka
from kafka import KafkaProducer
import json



In [ ]:
#create producer connection
producer= KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer= lambda v: json.dumps(v).encode("utf-8")
)

In [56]:
# create sensor event 

sensor_event= {
    "sensor_id": "S101",
    "temprature": 24.5,
    "humidity": 60
}

print(sensor_event)


{'sensor_id': 'S101', 'temprature': 24.5, 'humidity': 60}


In [57]:
 
# send message to kafka 
future = producer.send(
    "sensor-telemetry",
    key= b"sensor_id",
    value= sensor_event
)

print("message sent")


message sent


In [61]:
# wait for kafka confirmation
record_metadata= future.get(timeout=10)
print(record_metadata)

RecordMetadata(topic='sensor-telemetry', partition=0, topic_partition=TopicPartition(topic='sensor-telemetry', partition=0), offset=5, timestamp=1781712937328, checksum=None, serialized_key_size=9, serialized_value_size=57, serialized_header_size=-1)


In [64]:
sensor_event= [
{"sensor_id": "S101","temprature": 24},
{"sensor_id": "S102","temprature": 56},
{"sensor_id": "S103","temprature": 28},
{"sensor_id": "S104","temprature": 29},
{"sensor_id": "S105","temprature": 24},
{"sensor_id": "S106","temprature": 56},
{"sensor_id": "S107","temprature": 28},
{"sensor_id": "S108","temprature": 29}
]

for event in sensor_event:
    metadata= producer.send(
        "sensor-telemetry",
        key= event["sensor_id"].encode("utf-8"),
        value= event
    ).get()

    print(
        f"partition={metadata}"
        f"offset={metadata.offset}"
    )


partition=RecordMetadata(topic='sensor-telemetry', partition=0, topic_partition=TopicPartition(topic='sensor-telemetry', partition=0), offset=20, timestamp=1781713690527, checksum=None, serialized_key_size=4, serialized_value_size=39, serialized_header_size=-1)offset=20
partition=RecordMetadata(topic='sensor-telemetry', partition=1, topic_partition=TopicPartition(topic='sensor-telemetry', partition=1), offset=12, timestamp=1781713690534, checksum=None, serialized_key_size=4, serialized_value_size=39, serialized_header_size=-1)offset=12
partition=RecordMetadata(topic='sensor-telemetry', partition=0, topic_partition=TopicPartition(topic='sensor-telemetry', partition=0), offset=21, timestamp=1781713690538, checksum=None, serialized_key_size=4, serialized_value_size=39, serialized_header_size=-1)offset=21
partition=RecordMetadata(topic='sensor-telemetry', partition=0, topic_partition=TopicPartition(topic='sensor-telemetry', partition=0), offset=22, timestamp=1781713690541, checksum=None, s

In [65]:
producer.flush()
print("all message sent")

all message sent


In [18]:
#create our first consumer 
from kafka import KafkaConsumer

In [19]:
import  json
consumer= KafkaConsumer(
    "sensor-telemetry",
    bootstrap_servers= "localhost:9092",
    auto_offset_reset="earliest",
    value_deserializer= lambda m: json.loads(m.decode("utf-8")),
    consumer_timeout_ms=500
)

print(consumer)

In [20]:
## read message 
for msg in consumer:
    print(msg)
    

ConsumerRecord(topic='sensor-telemetry', partition=0, leader_epoch=0, offset=0, timestamp=1781621064745, timestamp_type=0, key=None, value={'senesor_id': 'S101', 'temprature': 24.5, 'humidity': 60}, headers=[], checksum=None, serialized_key_size=-1, serialized_value_size=58, serialized_header_size=-1)
ConsumerRecord(topic='sensor-telemetry', partition=0, leader_epoch=0, offset=1, timestamp=1781711657003, timestamp_type=0, key=None, value={'senesor_id': 'S103', 'temprature': 45}, headers=[], checksum=None, serialized_key_size=-1, serialized_value_size=40, serialized_header_size=-1)
ConsumerRecord(topic='sensor-telemetry', partition=0, leader_epoch=0, offset=2, timestamp=1781712361497, timestamp_type=0, key=None, value={'senesor_id': 'S104', 'temprature': 70}, headers=[], checksum=None, serialized_key_size=-1, serialized_value_size=40, serialized_header_size=-1)
ConsumerRecord(topic='sensor-telemetry', partition=0, leader_epoch=0, offset=3, timestamp=1781712365375, timestamp_type=0, key=

In [16]:
for msg in consumer:
    print(
        f"""
topic : {msg.topic}
partition : {msg.partition}
offset : {msg.offset}
value : {msg.value}
"""
)

In [21]:
#check consumer postion 
for partition in consumer.assignment():
    position = consumer.position(partition)
    print(f"{partition} -> next offset = {position}")

TopicPartition(topic='sensor-telemetry', partition=0) -> next offset = 24
TopicPartition(topic='sensor-telemetry', partition=1) -> next offset = 14
TopicPartition(topic='sensor-telemetry', partition=2) -> next offset = 6
